# 4.0 RNA-seq: Gene Expression in Diabetes

**A 2-hour class.** *All of Us* has measured **gene expression** (RNA sequencing, or RNA-seq) for thousands of participants. In this class we ask a simple question: **which genes are more or less active in people with diabetes than in people without it?**

### By the end of this notebook you'll be able to

1. Explain what RNA-seq measures, and the difference between **counts** and **TPM**
2. Find the *All of Us* RNA-seq files in your workspace and describe what each one holds
3. Build a diabetes / no-diabetes cohort of participants who have RNA-seq data
4. Load, filter, and **normalize** a gene expression matrix
5. Use **PCA** to look at overall patterns in the data
6. Run a **differential expression** analysis with `limma`, and read a volcano plot and a results table

| | |
|---|---|
| ⏱️ **Time** | 120 minutes |
| 📂 **Uses** | the *All of Us* RNA-seq files (v9) and the Controlled Tier dataset |
| 💾 **Saves** | `rnaseq_cohort.csv`, `rnaseq_diabetes_results.csv` |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 15 |
| 2. Setup and a tour of the RNA-seq files | 15 |
| 3. Build the cohort | 20 |
| 4. Load the counts | 15 |
| 5. Normalize and explore (PCA) | 20 |
| 6. Differential expression | 25 |
| 7. Discuss and wrap up | 10 |
| *Optional:* look up eQTLs | *extra* |

**Before you start:**

- Run **0.0 setup**, so `aou_setup.R` has your bucket and dataset
- This notebook **doesn't** need the files from 1.1 or 2.0: it builds its own cohort
- 👩‍🏫 **Instructor tip:** run the **Setup** cells once before class. Installing the extra packages takes 5–10 minutes the first time.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### From genes to reads

Every cell has the same DNA, but each cell only **uses** some of its genes. To use a gene, the cell copies it into **RNA**. The more active a gene is, the more RNA copies it makes.

**RNA-seq** measures this. In the lab, the RNA from a sample is broken into short pieces and sequenced, producing millions of short **reads**. A computer then works out which gene each read came from.

### Counts

The result is a **count matrix**: one row per **gene**, one column per **person**, and each value is the number of reads from that gene in that person.

| | Person A | Person B | Person C |
|---|---|---|---|
| Gene 1 | 1,520 | 980 | 2,310 |
| Gene 2 | 0 | 3 | 1 |

### Why counts need **normalizing**

Raw counts can't be compared directly:

- **Sequencing depth:** if Person A was sequenced twice as deeply as Person B, *every* gene will have about twice as many reads, even if nothing biological differs.
- **Gene length:** a long gene produces more reads than a short gene with the same activity.

*All of Us* provides two kinds of values:

| | What it is | Use it for |
|---|---|---|
| **Counts** (`gene_reads`) | raw read counts, whole numbers | **comparing groups** (differential expression): the statistical methods expect counts |
| **TPM** (`gene_tpm`) | *transcripts per million*: adjusted for depth **and** gene length, so each person's values add up to 1,000,000 | plots, and comparing genes within a person |

In this class we use **counts**, and normalize them for sequencing depth ourselves.

### Log scale

Expression values are very **skewed**: a few genes have huge counts, most have small ones. We work on the **log₂ scale**, where each step of 1 means "twice as much". A difference of **1** on the log₂ scale is a **2-fold** difference; **−1** is half as much.

### Differential expression

For **each gene**, we compare its average expression in people **with** diabetes to people **without**, while adjusting for **age** and **sex**. That's one regression model per gene: about 15,000 models.

- **log fold change (logFC):** the difference between the groups on the log₂ scale. logFC = 1 means twice as high in people with diabetes; logFC = −0.5 means about 30% lower.
- **Multiple testing:** with 15,000 tests, about 750 would have p < 0.05 **by chance alone**. So we use the **adjusted p-value** (the *false discovery rate*, FDR). Genes with **adjusted p < 0.05** are our findings: we expect fewer than 5% of them to be false positives.

## Part 2. Setup

This notebook needs three extra packages:

| Package | What it does |
|---|---|
| **data.table** | reads very large text files quickly (`fread()`) |
| **edgeR** | stores count data and normalizes it |
| **limma** | runs the differential expression models |

edgeR and limma come from **Bioconductor**, a collection of R packages for biology, so they're installed with `BiocManager::install()` instead of `install.packages()`. The cell below only installs what's missing.

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

# Extra packages for this notebook. First time only: this can take 5-10 minutes.
user_lib <- Sys.getenv("R_LIBS_USER")
dir.create(user_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(user_lib, .libPaths()))

for (pkg in c("data.table", "BiocManager")) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, lib = user_lib, repos = "https://cloud.r-project.org")
  }
}
for (pkg in c("limma", "edgeR")) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    BiocManager::install(pkg, lib = user_lib, update = FALSE, ask = FALSE)
  }
}

suppressPackageStartupMessages(library(edgeR))  # also loads limma

## Where are the RNA-seq files?

The *All of Us* data collection is **mounted** in your app as a folder under `~/workspace/`, just like your bucket. The RNA-seq files are inside it, in `v9/multiomics/rnaseq`.

**✏️ Check the folder name.** The data folder's name depends on your workspace. To see yours, run `list.files("~/workspace")` in a cell, and look for the one starting with `vwb-aou-datasets-controlled`. Edit `RNASEQ_DIR` below if yours is different.

In [ ]:
RNASEQ_DIR <- "~/workspace/vwb-aou-datasets-controlled-v8r9/v9/multiomics/rnaseq"

if (!dir.exists(RNASEQ_DIR)) stop("Folder not found. Run list.files('~/workspace') and edit RNASEQ_DIR.")

list.files(RNASEQ_DIR)
list.files(file.path(RNASEQ_DIR, "rnaseqc2"))

**🔍 Read the output:**

| File or folder | What's in it |
|---|---|
| `manifest.tsv` | one row per sample: its IDs and where its raw sequencing file is stored |
| `rnaseq_metadata.tsv` | information **about** each sample (for example, how and when it was processed) |
| `rnaseqc2/` | expression values from the **RNA-SeQC 2** pipeline: `gene_reads` (**counts**, what we use), `gene_tpm` (TPM), `exon_reads` (counts per exon), and `metrics` (quality checks) |
| `rsem/` | expression from a second pipeline, **RSEM**, including **transcript** (isoform) level values |
| `eqtl/`, `sqtl/` | precomputed results linking **genetic variants** to expression (eQTL) and to splicing (sQTL); see the optional section at the end |

The raw sequencing files (**BAM** files) are listed in the manifest. They're huge, and we don't need them for this class.

## The manifest

`read_tsv()` reads a tab-separated file. `col_types = cols(.default = "c")` reads every column as **text**, so long ID numbers are kept exactly as they are.

We only look at the **column names** and **counts**, not the rows: the rows are participant IDs, which we shouldn't display.

In [ ]:
manifest <- read_tsv(file.path(RNASEQ_DIR, "manifest.tsv"), col_types = cols(.default = "c"))

names(manifest)
nrow(manifest)                   # samples
n_distinct(manifest$research_id) # participants

**🔍 Read the output:** `research_id` is the participant's ID: the same number as `person_id` in the Controlled Tier dataset. `sampleid` is the ID of the sample. If the two counts are equal, everyone has one sample.

Now the sample metadata. Again, column names only:

In [ ]:
rna_metadata <- read_tsv(file.path(RNASEQ_DIR, "rnaseq_metadata.tsv"), col_types = cols(.default = "c"))

names(rna_metadata)

**💬 Discuss:** which of these columns describe the **sample processing** rather than the person (for example, a batch, plate, site, or date)? Differences like these can change expression values for technical reasons. We'll come back to this in Part 6.

## The count file

The counts are in a **GCT** file, a common format for expression data. It's a compressed (`.gz`) text file with:

- **Line 1:** the format version (`#1.2`)
- **Line 2:** the number of genes and the number of samples
- **Line 3:** the column names: `Name` (the gene's Ensembl ID), `Description` (the gene symbol), then **one column per sample**
- **Line 4 onward:** one row per gene

The full file is far too big to look at, so we read just the first 3 lines with `readLines()`.

In [ ]:
counts_file <- file.path(RNASEQ_DIR, "rnaseqc2", "aou_rnaseq_20260413.gene_reads.gct.gz")

gct_top <- readLines(gzfile(counts_file), n = 3)
gct_top[1:2]   # version, then genes and samples

file_columns <- str_split(gct_top[3], "\t")[[1]]
file_columns[1:2]          # the two gene columns
length(file_columns) - 2   # number of sample columns

**🔍 Read the output:** line 2 shows about **78,000 genes** and **9,000 samples**: roughly 700 million numbers. Loading all of it would need more memory than a standard app has. So in Part 4 we'll load **only the columns for the people in our cohort**.

The sample columns are named with an ID. This cell checks which manifest ID they use, then builds a lookup table from each **column name** to the **person_id**:

In [ ]:
sample_columns <- file_columns[-(1:2)]

c(columns_are_research_id = mean(sample_columns %in% manifest$research_id),
  columns_are_sampleid    = mean(sample_columns %in% manifest$sampleid))

column_id_type <- if (mean(sample_columns %in% manifest$research_id) > 0.9) "research_id" else "sampleid"

column_lookup <- manifest %>%
  transmute(column_name = .data[[column_id_type]], person_id = research_id) %>%
  filter(column_name %in% sample_columns) %>%
  distinct(person_id, .keep_all = TRUE)   # one sample per person

nrow(column_lookup)

**🔍 Read the output:** one of the two values should be **1** (100% of columns matched). `column_lookup` now has one row per participant with RNA-seq.

`.data[[column_id_type]]` means "the column whose name is stored in `column_id_type`". It lets the same code work whichever ID the file uses.

## Part 3. Build the cohort

We need to know, for each participant with RNA-seq data, their **age**, **sex at birth**, and whether they **have diabetes**. These come from the Controlled Tier dataset, using BigQuery.

**The diabetes definition is the same as in notebook 1.1:** anyone with a diagnosis of diabetes mellitus (concept **201820**) or any of its more specific types, found through the Cohort Builder's `cb_criteria` table. The SQL below copies that part of 1.1 unchanged.

What's new:

- `WHERE person.person_id IN ({rna_ids})` keeps **only participants with RNA-seq data**. `rna_ids` is the list of IDs, joined with commas.
- `CASE WHEN … THEN 'Yes' ELSE 'No' END` creates the **diabetes** column in one query: *Yes* if the person is in the diabetes list, *No* if not. (In 1.1, the same idea was split into two queries with `IN` and `NOT IN`.)

💰 This query costs a few cents, like the queries in 1.1.

In [ ]:
rna_ids <- str_c(column_lookup$person_id, collapse = ", ")

rna_cohort_sql <- r"---(
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        CASE WHEN person.person_id IN (SELECT
                criteria.person_id
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id
                FROM
                    `{CURRENT_DATASET}.cb_search_all_events`
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id
                    FROM
                        `{CURRENT_DATASET}.cb_criteria` c
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id
                        FROM
                            `{CURRENT_DATASET}.cb_criteria` cr
                        WHERE
                            concept_id IN (201820)
                            AND full_text LIKE '%_rank1]%'      ) a
                            ON (c.path LIKE CONCAT('%.', a.id, '.%')
                            OR c.path LIKE CONCAT('%.', a.id)
                            OR c.path LIKE CONCAT(a.id, '.%')
                            OR c.path = a.id)
                    WHERE
                        is_standard = 1
                        AND is_selectable = 1)
                    AND is_standard = 1 )) criteria )
        THEN 'Yes' ELSE 'No' END as diabetes
    FROM
        `{CURRENT_DATASET}.person` person
    LEFT JOIN
        `{CURRENT_DATASET}.concept` p_sex_at_birth_concept
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id
    WHERE
        person.person_id IN ({rna_ids})
)---"

rna_cohort <- run_query(str_glue(rna_cohort_sql)) %>%
  mutate(
    person_id = as.character(person_id),
    age       = as.numeric(interval(as_date(date_of_birth), today()) %/% years(1)),
    diabetes  = factor(diabetes, levels = c("No", "Yes"))
  ) %>%
  select(-date_of_birth)

save_data(rna_cohort, "rnaseq_cohort.csv")

glimpse(rna_cohort)

**🔍 Read the output:** one row per participant with RNA-seq data, with `age`, `sex_at_birth`, and `diabetes`.

- `interval(…) %/% years(1)` counts the **whole years** between the date of birth and today: the age.
- `factor(diabetes, levels = c("No", "Yes"))` makes **No** the first level, so it's the **reference group**: results will describe people **with** diabetes compared with people **without**.

We saved the cohort with `save_data()`. If your app restarts, you can skip the query and use `rna_cohort <- load_data("rnaseq_cohort.csv")` (then rerun the `mutate()` for `person_id` and `diabetes`).

## Describe the cohort

In [ ]:
rna_cohort %>%
  group_by(diabetes) %>%
  summarise(
    n          = n(),
    mean_age   = round(mean(age, na.rm = TRUE), 1),
    pct_female = round(100 * mean(sex_at_birth == "Female", na.rm = TRUE), 1)
  )

rna_cohort %>% count(sex_at_birth)

**🔍 Read the output:**

- How many participants with RNA-seq have diabetes?
- Are the groups similar in **age** and **sex**? If not, that's why we adjust for them.
- `sex_at_birth` may include answers other than *Female* and *Male* (for example, *Skip*). Our model needs a clear value, so we'll keep only *Female* and *Male*, and the groups have enough people for that.

> ⚠️ Remember: if any count in a table is between 1 and 20, don't share it.

## Choose the samples for this class

We'll take a **random sample of 300 people from each group**. This keeps the data small enough for a standard app and fast enough for class. With more time or a bigger app, you can increase `N_PER_GROUP` (see *Going further*).

- `set.seed()` makes the random sample **repeatable**: everyone who runs this gets the same people.
- `slice_sample(n = …)` picks that many rows at random **within each group** (because of `group_by()`). If a group has fewer people, it keeps all of them.

In [ ]:
N_PER_GROUP <- 300

set.seed(2026)
analysis_samples <- rna_cohort %>%
  filter(sex_at_birth %in% c("Female", "Male"), !is.na(age)) %>%
  group_by(diabetes) %>%
  slice_sample(n = N_PER_GROUP) %>%
  ungroup()

analysis_samples %>% count(diabetes)

## Part 4. Load the counts

To read **only our cohort's columns** from a 9,000-column file, we use two command-line tools that run before R sees the data:

| Tool | What it does here |
|---|---|
| `gzip -dc` | uncompresses the `.gz` file as it's read |
| `tail -n +3` | skips the first 2 lines (version and size), so the header is the first line |
| `cut -f 1,2,…` | keeps only the listed **column numbers**: the two gene columns and our samples |

`fread(cmd = …)` from **data.table** runs that command and reads the result. The pipe `|` passes the output of one tool to the next.

⏱️ This takes a few minutes: the whole file still has to be uncompressed once.

In [ ]:
keep <- column_lookup %>% filter(person_id %in% analysis_samples$person_id)
column_numbers <- match(keep$column_name, file_columns)   # positions in the file

read_command <- str_glue(
  "gzip -dc {shQuote(path.expand(counts_file))} | tail -n +3 | cut -f 1,2,{str_c(sort(column_numbers), collapse = ',')}"
)

counts_raw <- data.table::fread(cmd = read_command, sep = "\t", header = TRUE)
dim(counts_raw)

**🔍 Read the output:** about 78,000 rows (genes) and 600 + 2 columns.

Next we split it into:

- `genes`: the gene ID and symbol for each row
- `counts`: a **matrix** of numbers (genes × people). Matrices are what edgeR and limma work with.

Then we rename the columns to `person_id`, and put the sample table in the **same order as the columns**. `stopifnot()` stops with an error if the order doesn't match: a mismatch would silently give every person someone else's data.

In [ ]:
counts_raw <- as.data.frame(counts_raw)

genes  <- counts_raw[, c("Name", "Description")]
counts <- as.matrix(counts_raw[, -(1:2)])
rownames(counts) <- genes$Name
colnames(counts) <- keep$person_id[match(colnames(counts), keep$column_name)]

samples <- analysis_samples[match(colnames(counts), analysis_samples$person_id), ]
stopifnot(identical(samples$person_id, colnames(counts)))

rm(counts_raw)   # free up memory
dim(counts)

## Part 5. Normalize and explore

### Sequencing depth

`colSums()` adds up each column: the total reads for each person, also called the **library size**. We divide by 1,000,000 to show it in millions.

In [ ]:
summary(colSums(counts) / 1e6)   # millions of reads per person

mean(rowSums(counts) == 0)       # share of genes with no reads in anyone

**🔍 Read the output:**

- Library sizes **vary** between people, which is why we normalize.
- Many genes have **no reads at all**: many of the 78,000 "genes" are inactive in these samples. They carry no information and are removed next.

### Filter and normalize with edgeR

| Step | Function | What it does |
|---|---|---|
| 1 | `DGEList()` | stores the counts, the gene table, and the sample table together in one object |
| 2 | `filterByExpr()` | finds genes with enough reads, in enough people, to analyse |
| 3 | `calcNormFactors()` | calculates a **normalization factor** for each person (the *TMM* method), adjusting for depth and composition |
| 4 | `cpm(log = TRUE)` | converts to **log₂ counts per million (log-CPM)**: normalized values on the log scale, for plots and PCA |

`dge[keep_genes, , keep.lib.sizes = FALSE]` keeps only the chosen genes (rows) and all people (columns).

In [ ]:
dge <- DGEList(counts = counts, genes = genes, samples = samples)

keep_genes <- filterByExpr(dge, group = samples$diabetes)
table(keep_genes)

dge <- dge[keep_genes, , keep.lib.sizes = FALSE]
dge <- calcNormFactors(dge)

logcpm <- cpm(dge, log = TRUE)
dim(logcpm)

**🔍 Read the output:** `TRUE` is the number of genes we keep; usually around 15,000–20,000.

### Why the log scale?

Compare the distribution of the raw counts with the log-CPM values. Each histogram uses the values for one person.

In [ ]:
par(mfrow = c(1, 2))   # two plots side by side
hist(counts[keep_genes, 1], breaks = 100, main = "Raw counts", xlab = "reads")
hist(logcpm[, 1],           breaks = 50,  main = "log-CPM",    xlab = "log2 counts per million")
par(mfrow = c(1, 1))

**🔍 Read the output:** raw counts are squashed against zero with a long tail: a few genes have enormous counts. On the log scale, the values are spread out and much easier to work with.

### PCA: the big picture

Each person has ~15,000 expression values. **Principal component analysis (PCA)** summarises them into a few new variables, the **principal components (PCs)**:

- **PC1** is the direction in which people differ the **most**; **PC2** the next most, and so on
- People with similar expression profiles appear **close together** on a PC1-vs-PC2 plot
- Each PC explains some **percentage of the variation**

We use the 1,000 **most variable** genes, as is common. `prcomp()` runs the PCA; `t()` **transposes** the matrix (flips rows and columns), because `prcomp()` wants one row per person.

In [ ]:
gene_variance <- apply(logcpm, 1, var)            # variance of each gene (row)
top_genes     <- order(gene_variance, decreasing = TRUE)[1:1000]

pca <- prcomp(t(logcpm[top_genes, ]))
pct <- round(100 * pca$sdev^2 / sum(pca$sdev^2), 1)  # % variation per PC

pca_df <- tibble(
  PC1          = pca$x[, 1],
  PC2          = pca$x[, 2],
  diabetes     = samples$diabetes,
  sex_at_birth = samples$sex_at_birth,
  age          = samples$age
)

ggplot(pca_df, aes(PC1, PC2, colour = diabetes)) +
  geom_point(alpha = 0.6) +
  labs(x = str_glue("PC1 ({pct[1]}%)"), y = str_glue("PC2 ({pct[2]}%)"),
       title = "PCA of gene expression")

**🔍 Read the output:** do people with and without diabetes form separate clusters? Usually they **don't**: diabetes changes some genes a little, but it isn't the main source of variation. The big patterns in RNA-seq data often come from **technical** factors (how and when samples were processed, also called **batch effects**) or from **sex** and **cell composition**.

That's why we don't stop at PCA: Part 6 tests each gene individually.

### ✍️ Your turn

Colour the PCA plot by **sex at birth** instead. Do the groups separate more than they did by diabetes? Why might that be?

<details>
<summary><b>Show answer</b></summary>

```r
ggplot(pca_df, aes(PC1, PC2, colour = sex_at_birth)) +
  geom_point(alpha = 0.6)
```

Some genes are on the **X and Y chromosomes**, so their expression differs strongly by sex (for example, *XIST*, and genes on the Y chromosome that are only expressed by people with a Y chromosome). If these genes are among the 1,000 most variable, sex can show up clearly in a PC. If not, you may see little separation in PC1 and PC2: try `PC3` or `PC4` (`pca$x[, 3]`).

</details>

In [ ]:
# Your code here

## Part 6. Differential expression

We'll fit, for **every gene**, a linear model:

> expression ~ diabetes + age + sex at birth

and test the **diabetes** term. We use **limma**, which does this for all genes at once and borrows information across genes to make each test more reliable.

| Step | Function | What it does |
|---|---|---|
| 1 | `model.matrix()` | builds the **design matrix**: the model's variables as columns of numbers (like `glm()` does behind the scenes) |
| 2 | `voom()` | turns the counts into log-CPM values **with a weight** for each value, because low counts are less precise than high counts |
| 3 | `lmFit()` | fits the linear model to every gene |
| 4 | `eBayes()` | calculates the t-statistics and p-values, sharing information across genes |
| 5 | `topTable()` | returns a results table for one term (`diabetesYes`), sorted by p-value |

In [ ]:
design <- model.matrix(~ diabetes + age + sex_at_birth, data = samples)
colnames(design)

v   <- voom(dge, design)
fit <- eBayes(lmFit(v, design))

results <- topTable(fit, coef = "diabetesYes", number = Inf)

results %>%
  select(gene = Description, logFC, AveExpr, P.Value, adj.P.Val) %>%
  head(20)

**🔍 Read the output:**

| Column | Means |
|---|---|
| `gene` | the gene symbol |
| `logFC` | log₂ fold change, **diabetes vs no diabetes**. Positive = higher in people with diabetes; 1 = twice as high |
| `AveExpr` | the gene's average log-CPM: how active it is overall |
| `P.Value` | the p-value for this gene alone |
| `adj.P.Val` | the **adjusted p-value** (FDR), which accounts for testing ~15,000 genes. **Use this one.** |

`colnames(design)` shows the model's terms: `diabetesYes` is diabetes compared with the reference (*No*); `sex_at_birthMale` is Male compared with Female.

### How many genes are significant?

In [ ]:
results %>%
  mutate(direction = if_else(logFC > 0, "higher with diabetes", "lower with diabetes")) %>%
  filter(adj.P.Val < 0.05) %>%
  count(direction)

### Volcano plot

A **volcano plot** shows every gene at once:

- **x-axis:** the size of the difference (logFC). Further from 0 = bigger difference.
- **y-axis:** −log₁₀(p-value). Higher = stronger evidence. A p-value of 0.001 is 3; 0.000001 is 6.
- Genes **high and far to the left or right** are both large **and** convincing differences.

In [ ]:
ggplot(results, aes(logFC, -log10(P.Value), colour = adj.P.Val < 0.05)) +
  geom_point(size = 0.8, alpha = 0.6) +
  scale_colour_manual(values = c(`FALSE` = "grey60", `TRUE` = "firebrick"),
                      name = "Adjusted p < 0.05") +
  geom_vline(xintercept = 0, linetype = "dashed") +
  labs(x = "log2 fold change (diabetes vs no diabetes)", y = "-log10(p-value)",
       title = "Differential expression: diabetes")

### Look at the top gene

A boxplot of one gene's log-CPM values in each group. The box shows the middle half of the people; the line is the median.

In [ ]:
top_gene <- results$Name[1]

tibble(logCPM = logcpm[top_gene, ], diabetes = samples$diabetes) %>%
  ggplot(aes(diabetes, logCPM)) +
  geom_boxplot() +
  labs(title = results$Description[1], y = "log2 counts per million")

**🔍 Read the output:** the groups overlap a lot, even for the most significant gene. Differences in expression between groups of people are usually **small shifts in the average**, not on/off switches. With hundreds of people, small shifts can still be clearly significant.

### Save the results

The results table has one row per **gene** (no participant data), so it's safe to save.

In [ ]:
save_data(results, "rnaseq_diabetes_results.csv")

### ✍️ Your turn

1. The gene ***TXNIP*** is known to respond to blood sugar. Find it in `results`. Is it higher or lower in people with diabetes? Is it significant?
2. Look back at `names(rna_metadata)`. Suppose it had a column called `batch`. How would you add it to the model? (You don't need to run it.)

<details>
<summary><b>Show answer</b></summary>

1.
```r
results %>% filter(Description == "TXNIP") %>%
  select(gene = Description, logFC, adj.P.Val)
```
A positive `logFC` means higher in people with diabetes. It's significant if `adj.P.Val` is below 0.05.

2. Join the metadata to `samples` (matching on the participant's ID), then add the column to the formula:
```r
design <- model.matrix(~ diabetes + age + sex_at_birth + batch, data = samples)
```
Adjusting for batch removes technical differences between processing batches, the same way we adjust for age and sex.

</details>

In [ ]:
# Your code here

## Part 7. Discuss

1. **Cause or effect?** If a gene is higher in people with diabetes, did it **cause** the diabetes, or is it a **result** of diabetes (or of high blood sugar)?
2. **Medications:** many people with diabetes take drugs like metformin. Could a drug, rather than the disease, explain some differences? How could you check?
3. **Other confounders:** we adjusted for age and sex. What else differs between the groups and affects gene expression? (Think of BMI, other conditions, and the mix of cell types in the sample.)
4. **Definition of diabetes:** our flag means "ever had a diabetes diagnosis in the health records". It doesn't say **when**, or whether the RNA sample was taken before or after the diagnosis. Why might that matter?

## ⚠️ Common errors

| Message | Means |
|---|---|
| *Folder not found* | Edit `RNASEQ_DIR`: run `list.files("~/workspace")` to find the data folder's name |
| *there is no package called 'edgeR'* | Rerun the Setup cell and wait for the install to finish |
| *cannot allocate vector of size …* | The app ran out of memory: lower `N_PER_GROUP`, or use an app with more memory |
| *stopifnot(…) is not TRUE* | The sample table and the count columns don't match. Rerun Part 4 from the start. |
| *object 'samples' not found* | Run the cells in order; Part 4 creates `samples` |

## ✅ Key takeaways

- RNA-seq counts how many reads come from each gene in each person
- Use **counts** for comparing groups; **normalize** for sequencing depth, and work on the **log scale**
- **PCA** shows the main patterns in the data, which are often technical, not the disease
- **Differential expression** fits one model per gene; read the **logFC** and the **adjusted p-value**
- Results are **associations**: think about confounders, medications, and timing

**Going further:**

- Increase `N_PER_GROUP` (or use everyone) for more power. Use an app with more memory, and expect the loading step to take longer.
- Add more covariates from the Controlled Tier dataset, such as BMI, to the model
- Look up your top genes in the optional eQTL section below

## Optional: are your genes controlled by genetic variants?

An **eQTL** (*expression quantitative trait locus*) is a genetic variant linked to how active a gene is. *All of Us* has already calculated eQTLs for each gene, separately for each genetic ancestry group and for everyone combined (**COMB**).

The file below has **one row per gene**: the gene's strongest nearby variant. Let's check whether our significant genes have an eQTL.

Column names can differ between releases, so we first look at them. The code assumes the gene ID column is called `phenotype_id`; if it isn't, edit that line.

In [ ]:
list.files(file.path(RNASEQ_DIR, "eqtl", "ciseqtl"))

eqtl <- read_tsv(file.path(RNASEQ_DIR, "eqtl", "ciseqtl", "COMB.cis_qtl.txt.gz"),
                 show_col_types = FALSE)
names(eqtl)

In [ ]:
# Gene IDs sometimes end in a version number (ENSG00000123456.7); remove it before matching
significant <- results %>%
  filter(adj.P.Val < 0.05) %>%
  mutate(gene_id = str_remove(Name, "\\..*")) %>%
  select(gene_id, gene = Description, logFC, adj.P.Val)

eqtl %>%
  mutate(gene_id = str_remove(phenotype_id, "\\..*")) %>%
  inner_join(significant, by = "gene_id") %>%
  select(gene, logFC, adj.P.Val, variant_id, pval_beta) %>%
  arrange(pval_beta) %>%
  head(20)

**🔍 Read the output:** each row is one of our diabetes-related genes and its strongest eQTL variant. A very small `pval_beta` means the variant clearly affects that gene's expression. A gene that is both linked to diabetes **and** controlled by a genetic variant is a lead for further study: for example, is that variant also linked to diabetes in a GWAS?

**Next:** clear all outputs before you share or save this notebook to GitHub (**Edit → Clear Outputs of All Cells**).